In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install imagehash

import os
from PIL import Image, UnidentifiedImageError
import imagehash
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn
import torch.optim as optim

# =======================
# 1️⃣ Paths
# =======================
data_dir = '/content/drive/MyDrive/grape'  # original dataset
clean_dir = '/content/drive/MyDrive/grape_cleaned'  # cleaned images
os.makedirs(clean_dir, exist_ok=True)

# =======================
# 2️⃣ Remove duplicates & corrupt images & multi-leaf/text images
# =======================
hashes = {}

import os
from PIL import Image, UnidentifiedImageError
import imagehash

hashes = {}

for root, _, files in os.walk(data_dir):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            img = Image.open(file_path)
            img.verify()  # check corrupt
            img = Image.open(file_path)  # reopen for hashing

            # Convert to RGB to handle RGBA / grayscale issues
            if img.mode != 'RGB':
                img = img.convert('RGB')

            # Compute perceptual hash
            h = imagehash.phash(img)
            if h in hashes:
                print(f"Duplicate skipped: {file_path}")
                continue
            else:
                hashes[h] = file_path

        except (UnidentifiedImageError, OSError, SyntaxError):
            print(f"Corrupt skipped: {file_path}")
            continue

        # Skip mostly empty images or text (grayscale extrema check)
        img_gray = img.convert("L")
        extrema = img_gray.getextrema()
        if extrema[1] - extrema[0] < 10:
            print(f"Empty/text image skipped: {file_path}")
            continue

        # Copy cleaned image to new folder with .jpg extension
        rel_path = os.path.relpath(file_path, data_dir)
        base_name = os.path.splitext(rel_path)[0] + ".jpg"  # force .jpg
        new_path = os.path.join(clean_dir, base_name)
        os.makedirs(os.path.dirname(new_path), exist_ok=True)

        # Save as JPEG
        img.save(new_path, format='JPEG')

print("✅ Dataset cleaning completed!")

# =======================
# 3️⃣ Aggressive Data Transform
# =======================
transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(224, scale=(0.7,1.0)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
    transforms.ToTensor()
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# =======================
# 4️⃣ Prepare Dataset
# =======================
full_dataset = datasets.ImageFolder(root=clean_dir, transform=transform_train)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Replace val transform
val_dataset.dataset.transform = transform_val

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =======================
# 5️⃣ EfficientNet Transfer Learning
# =======================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

num_classes = len(full_dataset.classes)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, num_classes)
)
model = model.to(device)

# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# =======================
# 6️⃣ Training Loop
# =======================
num_epochs = 15
best_acc = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    correct, total = 0,0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs,1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)

    val_acc = correct/total
    avg_loss = running_loss/len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f} Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_cotton_model.pth")
        print(f"✅ Saved Best Model with Acc: {best_acc:.4f}")

# =======================
# 7️⃣ Inference Function
# =======================
def predict(image_path):
    image = Image.open(image_path)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    image = transform_val(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs,1)
    return full_dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 9.8 MB/s eta 0:00:00
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grape_blackrot_29.jpg
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grape_blackrot_54.jpg
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grape_blackrot_67.jpg
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grape_blackrot_72.jpg
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grape_blackrot_73.jpg
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grape_blackrot_92.jpg
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grape_blackrot_94.jpg
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grape_blackrot_105.jpg
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grape_blackrot_107.jpg
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grape_blackrot_122.jpg
Duplicate skipped: /content/drive/MyDrive/grape/grape_blackrot/grap

100%|██████████| 20.5M/20.5M [00:00<00:00, 142MB/s]


Epoch [1/15] Loss: 2.1225 Val Acc: 0.3855
✅ Saved Best Model with Acc: 0.3855
Epoch [2/15] Loss: 1.7340 Val Acc: 0.4618
✅ Saved Best Model with Acc: 0.4618
Epoch [3/15] Loss: 1.4391 Val Acc: 0.5060
✅ Saved Best Model with Acc: 0.5060
Epoch [4/15] Loss: 1.2619 Val Acc: 0.4900
Epoch [5/15] Loss: 1.1661 Val Acc: 0.4900
Epoch [6/15] Loss: 1.0100 Val Acc: 0.5060
Epoch [7/15] Loss: 0.9315 Val Acc: 0.5020
Epoch [8/15] Loss: 0.8594 Val Acc: 0.5301
✅ Saved Best Model with Acc: 0.5301
Epoch [9/15] Loss: 0.8007 Val Acc: 0.5020
Epoch [10/15] Loss: 0.7887 Val Acc: 0.5141
Epoch [11/15] Loss: 0.8511 Val Acc: 0.5221
Epoch [12/15] Loss: 0.7908 Val Acc: 0.5221
Epoch [13/15] Loss: 0.8266 Val Acc: 0.5221
Epoch [14/15] Loss: 0.7970 Val Acc: 0.5382
✅ Saved Best Model with Acc: 0.5382
Epoch [15/15] Loss: 0.7952 Val Acc: 0.5301


In [ ]:
import os
from PIL import Image
from torchvision import transforms
import random

clean_dir = '/content/drive/MyDrive/grape_cleaned'
balanced_dir = '/content/drive/MyDrive/grape_balanced'
os.makedirs(balanced_dir, exist_ok=True)

# Aggressive augmentation pipeline
augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
])

target_count = 500  # desired number of images per class

for class_name in os.listdir(clean_dir):
    class_path = os.path.join(clean_dir, class_name)
    if not os.path.isdir(class_path):
        continue

    balanced_class_path = os.path.join(balanced_dir, class_name)
    os.makedirs(balanced_class_path, exist_ok=True)

    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    num_images = len(images)

    # Copy original images first
    for f in images:
        src = os.path.join(class_path, f)
        dst = os.path.join(balanced_class_path, f)
        Image.open(src).save(dst)

    # Generate augmented images if needed
    while num_images < target_count:
        img_name = random.choice(images)
        img_path = os.path.join(class_path, img_name)
        img = Image.open(img_path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        aug_img = augment(img)
        new_name = f"{os.path.splitext(img_name)[0]}_aug_{num_images}.jpg"
        aug_img.save(os.path.join(balanced_class_path, new_name))
        num_images += 1

print("✅ Balanced dataset created!")


✅ Balanced dataset created!


In [ ]:
import os
from PIL import Image
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn
import torch.optim as optim

# =======================
# 1️⃣ Paths
# =======================
balanced_dir = '/content/drive/MyDrive/grape_balanced'  # balanced dataset

# =======================
# 2️⃣ Data Transforms
# =======================
transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10),
    transforms.ToTensor()
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# =======================
# 3️⃣ Prepare Dataset
# =======================
full_dataset = datasets.ImageFolder(root=balanced_dir, transform=transform_train)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Replace val transform
val_dataset.dataset.transform = transform_val

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =======================
# 4️⃣ EfficientNet-B0 Transfer Learning
# =======================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

num_classes = len(full_dataset.classes)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, num_classes)
)
model = model.to(device)

# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# =======================
# 5️⃣ Training Loop
# =======================
num_epochs = 10
best_acc = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs,1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f} Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_efficientnet_cucumber.pth")
        print(f"✅ Saved Best Model with Acc: {best_acc:.4f}")

# =======================
# 6️⃣ Inference Function
# =======================
def predict(image_path):
    image = Image.open(image_path)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    transform_val = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
    image = transform_val(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs,1)
    return full_dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


Epoch [1/10] Loss: 1.7779 Val Acc: 0.5090
✅ Saved Best Model with Acc: 0.5090
Epoch [2/10] Loss: 1.2846 Val Acc: 0.5409
✅ Saved Best Model with Acc: 0.5409
Epoch [3/10] Loss: 1.0476 Val Acc: 0.6267
✅ Saved Best Model with Acc: 0.6267
Epoch [4/10] Loss: 0.8579 Val Acc: 0.6567
✅ Saved Best Model with Acc: 0.6567
Epoch [5/10] Loss: 0.7042 Val Acc: 0.6906
✅ Saved Best Model with Acc: 0.6906
Epoch [6/10] Loss: 0.6004 Val Acc: 0.7056
✅ Saved Best Model with Acc: 0.7056
Epoch [7/10] Loss: 0.5154 Val Acc: 0.7106
✅ Saved Best Model with Acc: 0.7106
Epoch [8/10] Loss: 0.4816 Val Acc: 0.7216
✅ Saved Best Model with Acc: 0.7216
Epoch [9/10] Loss: 0.4604 Val Acc: 0.7246
✅ Saved Best Model with Acc: 0.7246
Epoch [10/10] Loss: 0.4427 Val Acc: 0.7255
✅ Saved Best Model with Acc: 0.7255


In [ ]:
import os

# Create folder if it doesn't exist
save_dir = "/content/drive/MyDrive/models"
os.makedirs(save_dir, exist_ok=True)

# Now save the model
torch.save(model.state_dict(), os.path.join(save_dir, "best_efficientnet_grape1.pth"))
print("✅ Model saved successfully!")


✅ Model saved successfully!
